# Combining artists line up through out the year

In [1]:
import pandas as pd
from pathlib import Path

# Look for extract folder relative to this notebook
path = Path("../data/extract")

# Load all per-year CSVs
dfs = [pd.read_csv(file) for file in path.glob("*_edc_lineup.csv")]
if len(dfs) == 0:
    raise FileNotFoundError(f"No CSV files found. Run the collection notebook first.")

# Combine new data
edc_new = pd.concat(dfs, ignore_index=True)
output_file = Path("../data/main/edc_all_artists.csv")

# Create directory if it doesn't exist
output_file.parent.mkdir(parents=True, exist_ok=True)

if output_file.exists():
    print(f"Reading existing data from {output_file}")
    edc_existing = pd.read_csv(output_file)
    
    # Combine and drop duplicates to avoid adding existing entries
    # dropping based on all columns to check if the entry exists
    edc_all = pd.concat([edc_existing, edc_new], ignore_index=True).drop_duplicates()
    
    print(f"Appended {len(edc_all) - len(edc_existing)} new unique rows")
else:
    print(f"Creating new file at {output_file}")
    edc_all = edc_new

edc_all.to_csv(output_file, index=False)

print(f"Total rows in {output_file}: {len(edc_all)}")
display(edc_all.head())

Reading existing data from ../data/main/edc_all_artists.csv
Appended 1 new unique rows
Total rows in ../data/main/edc_all_artists.csv: 1477


,year,artist
0,2024,1080p
1,2024,A Shade of Black
2,2024,Aaron K
3,2024,Abana
4,2024,D. Zeledon


# Normalize and strip artist names

In [2]:
# Normalize and strip artist names
artist_cols = [c for c in edc_all.columns if c.lower() == "artist"]
if not artist_cols:
    raise KeyError("No artist column found. Available columns: " + ", ".join(edc_all.columns))

artist_col = artist_cols[0]
edc_all["artist"] = (
    edc_all[artist_col]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace("&", "and")
)

before = len(edc_all)
edc_all = edc_all.drop_duplicates(subset=["year", "artist"]).reset_index(drop=True)
after = len(edc_all)
print(f"Deduped on [year, artist]: {before} → {after}")
display(edc_all.head())

Deduped on [year, artist]: 1477 → 1477


,year,artist
0,2024,1080p
1,2024,a shade of black
2,2024,aaron k
3,2024,abana
4,2024,d. zeledon


# Show how many times that artist played at EDC from 2022-2026

In [3]:
# Count how many times each artist appeared
if "year" not in edc_all.columns:
    raise KeyError("Column 'year' not found in edc_all. Ensure the combine step added the year column.")

artist_counts = (
    edc_all.groupby("artist").agg(
        total_appearances=("year", "count"),
        years_played=("year", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values("total_appearances", ascending=False)
)

min_year = int(edc_all['year'].min())
max_year = int(edc_all['year'].max())
print(f"Top 10 artists by appearances ({min_year}-{max_year}):")
display(artist_counts.head(10))

# Save outputs
edc_all_file = "../data/main/edc_all.csv"
artist_counts_file = f"../data/main/artist_counts_{min_year}_{max_year}.csv"

Path(edc_all_file).parent.mkdir(parents=True, exist_ok=True)

edc_all.to_csv(edc_all_file, index=False)
artist_counts.to_csv(artist_counts_file, index=False)

print(f"\nSaved combined data to {edc_all_file}")
print(f"Saved appearance counts to {artist_counts_file}")

Top 10 artists by appearances (2022-2025):


,artist,total_appearances,years_played
85,armnhmr,4,"[2022, 2023, 2024, 2025]"
164,bunny,4,"[2022, 2023, 2024, 2025]"
482,jessica audiffred,4,"[2022, 2023, 2024, 2025]"
556,lady faith,4,"[2022, 2023, 2024, 2025]"
84,armin van buuren,4,"[2022, 2023, 2024, 2025]"
648,matroda,4,"[2022, 2023, 2024, 2025]"
512,kaskade,4,"[2022, 2023, 2024, 2025]"
269,deorro,4,"[2022, 2023, 2024, 2025]"
27,adrenalize,4,"[2022, 2023, 2024, 2025]"
969,trouble,4,"[2022, 2023, 2024, 2025]"



Saved combined data to ../data/main/edc_all.csv
Saved appearance counts to ../data/main/artist_counts_2022_2025.csv


# Combine artists of from different agencies

In [4]:
import pandas as pd
from pathlib import Path

# Look for extract folder relative to this notebook
search_paths = [
    Path("../extract"),            # expected from notebook folder
    Path("edc_vegas/extract"),     # if run from repo root
    Path("../data/extract"),       # fallback if files were placed under data/extract
    Path("edc_vegas/data/extract")
]
path = next((p for p in search_paths if p.exists()), None)

if path is None:
    raise FileNotFoundError("No extract directory found. Checked: " + ", ".join(str(p) for p in search_paths))

# Load all per-year CSVs
dfs = [pd.read_csv(file) for file in path.glob("*_artists.csv")]
if len(dfs) == 0:
    raise FileNotFoundError(f"No CSV files found in {path}. Run the collection notebook first.")

# Combine and save
agency_all = pd.concat(dfs, ignore_index=True)

###########################################

# Normalize and strip artist names
artist_cols = [c for c in agency_all.columns if c.lower() == "artist"]
if not artist_cols:
    raise KeyError("No artist column found. Available columns: " + ", ".join(agency_all.columns))

print("Arttist names to lower case and strip:")
artist_col = artist_cols[0]
agency_all["artist"] = (
    agency_all[artist_col]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace("&", "and")
)
output_file = "../data/main/artists_agency.csv"
agency_all.to_csv(output_file, index=False)

print(f"Combined {len(agency_all)} rows → {output_file}")
display(agency_all.head())

Arttist names to lower case and strip:
Combined 9824 rows → ../data/main/artists_agency.csv


,artist,agency
0,nicole moudaber,insomniac
1,walker and royce,insomniac
2,virtual riot,insomniac
3,benwal,insomniac
4,ignez,insomniac
